# Pgvecto_rs embedding storage
In this notebook i show how you can connect pgvecto to llama_index and be able to query the db to get the embeddings stored

In [3]:
# i will set up the logger so the processing will be logged in the console to better understand what is happening and be able to debug properly

import logging
import sys
import os

logging.basicConfig(stream=sys.stdout, level=logging.INFO)
logging.getLogger().addHandler(logging.StreamHandler(stream=sys.stdout))

In [ ]:
from pgvecto_rs.sdk import PGVectoRs
from llama_index.vector_stores.pgvecto_rs import PGVectoRsStore

DATABASE_URL = "postgresql+psycopg://{username}:{password}@{host}:{port}/{db_name}".format(
    port=os.getenv("DB_PORT", "5432"),
    host=os.getenv("DB_HOST", "localhost"),
    username=os.getenv("DB_USER", "postgres"),
    password=os.getenv("DB_PASS", "mysecretpassword"),
    db_name=os.getenv("DB_NAME", "postgres"),
)

client = PGVectoRs(
    db_url=DATABASE_URL,
    collection_name="items",
    dimension=1536,  # Using OpenAI’s text-embedding-ada-002
)

vector_store = PGVectoRsStore(client=client)


In [ ]:
from llama_index.core import SimpleDirectoryReader
from llama_index.core.ingestion import IngestionPipeline, DocstoreStrategy
from llama_index.core.node_parser import SentenceSplitter
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.storage.docstore.postgres import PostgresDocumentStore

documents = SimpleDirectoryReader(input_files=["../data/paul_graham_essay.txt"])

# the splitter that will be used to create chunks from the doc
splitter = SentenceSplitter(
    chunk_size=520,
    chunk_overlap=50
)

# the model that will be used to make the embeddings from the nodes
embedder = OpenAIEmbedding(mode="text-embedding-ada-002")

pipeline = IngestionPipeline(
    transformations=[
        splitter,
        embedder,
    ],
    vector_store=vector_store,
    # i use the docstore to track files that have already been processed
    docstore=PostgresDocumentStore.from_uri(uri=DATABASE_URL, table_name="items_docstore"),
    docstore_strategy=DocstoreStrategy.UPSERTS # so that we update if the hash is different else we skip
)

pipeline.run(documents=[documents])


In [ ]:
from llama_index.core import VectorStoreIndex
# let us now try to quickly query the table

index = VectorStoreIndex.from_vector_store(vector_store=vector_store)

query_engine = index.as_query_engine()

response = query_engine.query("What would you say is the most important thing that the writer would want me to know?")